
# NaviGator Toolkit API test

This notebook is designed to test connection to the NaviGator Toolkit API ([see here for more information](https://it.ufl.edu/ai/navigator-toolkit/)). You will need a NaviGator API key. The key should be stored in a `.json` file with the following format:

    {
      "OPENAI_API_KEY" : "Put your key here in the quotes",
      "base_url" : "https://api.ai.it.ufl.edu/"
    }

We suggest putting that in your home directory, to minimize the chances of accidentally adding and committing the file to a git repo. Remember that anyone with your API key can use NaviGator as you! 

In [3]:
import openai
import os
import json
from pathlib import Path

## Load `.json` file with your key and API endpoint URL

In [7]:
# Set the path to your JSON key file
key_file = Path.home() / 'navigator_api_keys.json'

# Load the JSON file
with open(key_file, 'r') as file:
    data = json.load(file)

# Extract the values
OPENAI_API_KEY = data.get('OPENAI_API_KEY')
base_url = data.get('base_url')

# Set the environment variable
os.environ['TOOLKIT_API_KEY'] = OPENAI_API_KEY


## Test connectivity and get model list

Reply should list the models that are available with your API key. An example, truncated, output is:

    SyncPage[Model](data=[Model(id='llama-3.1-70b-instruct', created=1677610602, object='model', owned_by='openai'), Model(id='sfr-embedding-mistral', created=1677610602, object='model', owned_by='openai'),...)], object='list')

In [8]:
# Check list of available models
client = openai.OpenAI(
    api_key=os.environ.get("TOOLKIT_API_KEY"),
    base_url=base_url
)

response = client.models.list()
 
print(response)

SyncPage[Model](data=[Model(id='flux.1-dev', created=1677610602, object='model', owned_by='openai'), Model(id='flux.1-schnell', created=1677610602, object='model', owned_by='openai'), Model(id='flux.2-klein', created=1677610602, object='model', owned_by='openai'), Model(id='gemma-4-31b-it', created=1677610602, object='model', owned_by='openai'), Model(id='gpt-oss-120b', created=1677610602, object='model', owned_by='openai'), Model(id='gpt-oss-20b', created=1677610602, object='model', owned_by='openai'), Model(id='granite-3.3-8b-instruct', created=1677610602, object='model', owned_by='openai'), Model(id='kokoro', created=1677610602, object='model', owned_by='openai'), Model(id='llama-3.3-70b-instruct', created=1677610602, object='model', owned_by='openai'), Model(id='medgemma-27b-it', created=1677610602, object='model', owned_by='openai'), Model(id='meta-muse-glimmer-30b', created=1677610602, object='model', owned_by='openai'), Model(id='mistral-small-3.1', created=1677610602, object='m

In [9]:
# Print available models in better format
for model in response:
    print(model.id)

flux.1-dev
flux.1-schnell
flux.2-klein
gemma-4-31b-it
gpt-oss-120b
gpt-oss-20b
granite-3.3-8b-instruct
kokoro
llama-3.3-70b-instruct
medgemma-27b-it
meta-muse-glimmer-30b
mistral-small-3.1
nemotron-3-nano-30b-a3b
nemotron-3-super-120b-a12b
nomic-embed-text-v1.5
sfr-embedding-mistral
whisper-large-v3


## Test model completion

You may need to update the model from the list above as the available models change over time.

Here is an example response. Remember that LLM generation is stochastic, so you should not expect to get the same reply, but something similar.
   
       ChatCompletion(id='chat-644e583f079c4c33a1e95480caca4f99', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Here's a short poem about ice cream:\n\nA cone in my hand, oh so sweet,\nA treat that can't be beat.\nThe flavors dance on my tongue so bright,\nA delightful treat on a summer night.\n\nThe colors and sprinkles, a joyful sight,\nA smile on my face, pure delight.\nThe cold and creamy texture, a pleasure to share,\nA sweet indulgence, beyond compare.\n\nSo here's to ice cream, a treat so fine,\nA sweet escape, that's simply divine.", refusal=None, role='assistant', function_call=None, tool_calls=None))], created=1739560733, model='meta-llama/Llama-3.1-70B-Instruct', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=104, prompt_tokens=45, total_tokens=149, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None)

In [10]:
# Set model from list above
model = "nemotron-3-nano-30b-a3b"

response = client.chat.completions.create(
    model=model,
    messages = [
        {
            "role": "user",
            "content": "Write a short poem about something fun."
        }
    ]
)
 
print(response)


ChatCompletion(id='chatcmpl-a4fbc50a8a7e4a36', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n**Bubble‑Pop Ballet**\n\nIn a kitchen of cheer, the soda pops prance,  \nTwisting on spoons in a giggle‑filled trance.  \nThey spin like confetti, they swoosh, they say “yes!”  \nTo the rhythm of laughter that bubbles and bless.\n\nJust a splash of lemon, a tickle of lime—  \nA tiny parade on the tip of your rhyme.  \nPop them, roll them, let the bubbles sing,  \nAnd dance through the day on a fun‑filled wing.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='User wants a short poem about something fun. Simple. Should be creative.\n', provider_specific_fields={'refusal': None, 'reasoning': 'User wants a short poem about something fun. Simple. Should be creative.\n', 'reasoning_content': 'User wants a short poem about something fun. Simple. Should be creative.\n'}), prov

In [11]:
# Print nicely formatted message

print(response.choices[0].message.content)


**Bubble‑Pop Ballet**

In a kitchen of cheer, the soda pops prance,  
Twisting on spoons in a giggle‑filled trance.  
They spin like confetti, they swoosh, they say “yes!”  
To the rhythm of laughter that bubbles and bless.

Just a splash of lemon, a tickle of lime—  
A tiny parade on the tip of your rhyme.  
Pop them, roll them, let the bubbles sing,  
And dance through the day on a fun‑filled wing.


## Test sending a system prompt and content

In [13]:
# Set model from list above
model = 'nemotron-3-nano-30b-a3b'

response = client.chat.completions.create(
    model=model, # model to send to the proxy
    messages = [
       {
         "role": "system",
         "content": "You are a poetic assistant, skilled in explaining complex programming concepts with creative flair."
      },
       {
         "role": "user",
         "content": "what is the largest galaxy?"
       }
     ]

)
 
print(response)

ChatCompletion(id='chatcmpl-9b41e28041b270e9', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n**The gigantic\u202fIC\u202f1101 – a galaxy that stretches the imagination as far as a galaxy‑spanning road trip**\n\nMost of us think of galaxies as glittering swirls of stars, dust and gas, but some of them grow so vast that they make even the most sprawling continents look like tiny towns.  \nWhen astronomers ask “what is the *largest* galaxy?” they usually mean **the one with the biggest *physical diameter***—the distance you would need to travel, at the speed of light, to go from one edge to the other.\n\n### The current record‑holder: **IC\u202f1101**\n\n- **Location:** In the heart of the galaxy cluster Abell\u202f2029, about **1.04 billion light‑years** from us.  \n- **Size:** Roughly **6\u202fmillion light‑years** across. If you placed it at the centre of our Milky Way, it would extend well beyond the orbit of Pluto—out to the di

In [14]:
# Print nicely formatted message

print(response.choices[0].message.content)


**The gigantic IC 1101 – a galaxy that stretches the imagination as far as a galaxy‑spanning road trip**

Most of us think of galaxies as glittering swirls of stars, dust and gas, but some of them grow so vast that they make even the most sprawling continents look like tiny towns.  
When astronomers ask “what is the *largest* galaxy?” they usually mean **the one with the biggest *physical diameter***—the distance you would need to travel, at the speed of light, to go from one edge to the other.

### The current record‑holder: **IC 1101**

- **Location:** In the heart of the galaxy cluster Abell 2029, about **1.04 billion light‑years** from us.  
- **Size:** Roughly **6 million light‑years** across. If you placed it at the centre of our Milky Way, it would extend well beyond the orbit of Pluto—out to the distant realm where the Sun’s light would still be a faint whisper.  
- **Mass:** An astonishing **~2 × 10¹³ M☉** (about two trillion times the mass of our Sun).  
- **What it looks li